# Normalizers Deep Dive

Normalizers clean and standardize field values before graph construction — ensuring
consistent node properties, accurate matching, and reliable queries.

| Normalizer | Purpose |
|------------|--------|
| NormalizeWhitespace | Strip leading/trailing whitespace, collapse internal spaces |
| NormalizeNulls | Convert '', 'N/A', 'null', None to consistent null representation |
| NormalizeCase | Standardize to upper/lower/title case |
| NormalizeEnum | Map variant values to canonical enum labels |
| NormalizeSpelling | Correct common misspellings in field values |
| NormalizeTimestamp | Parse varied date formats into ISO 8601 |

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph` (base install)
- No Neptune or AWS credentials required


In [ ]:
from graphrag_toolkit.document_graph.transform.transformer_provider_config import TransformerProviderConfig
from graphrag_toolkit.document_graph.transform.normalizers.normalize_whitespace_provider import NormalizeWhitespaceProvider
from graphrag_toolkit.document_graph.transform.normalizers.normalize_nulls_provider import NormalizeNullsProvider
from graphrag_toolkit.document_graph.transform.normalizers.normalize_case_provider import NormalizeCaseProvider
from graphrag_toolkit.document_graph.transform.normalizers.normalize_enum_provider import NormalizeEnumProvider
from graphrag_toolkit.document_graph.transform.normalizers.normalize_spelling_provider import NormalizeSpellingProvider
from graphrag_toolkit.document_graph.transform.normalizers.normalize_timestamp_provider import NormalizeTimestampProvider

## Whitespace Normalizer

Strip leading/trailing whitespace and collapse multiple internal spaces.
Prevents duplicate nodes caused by invisible whitespace differences (e.g., `"Alice"` vs `" Alice "`).


In [ ]:
records = [
    {'id': 'C-001', 'title': '  Access   Control  Policy  ', 'description': 'Ensure\t\tall  users   have  MFA'},
    {'id': 'C-002', 'title': '  Data    Encryption ', 'description': 'Encrypt   data  at   rest  and in  transit'}
]
config = TransformerProviderConfig(name='normalize_ws', args={'fields': ['title', 'description']})
transformer = NormalizeWhitespaceProvider(config)
result = transformer.transform(records)
print("NormalizeWhitespaceProvider:")
for r in result:
    print(f"  title: '{r['title']}'")
    print(f"  desc:  '{r['description']}'")

## Null Normalizer

Convert inconsistent null representations (`''`, `'N/A'`, `'null'`, `'None'`, `None`) to a
single consistent format. Prevents properties with meaningless string values cluttering the graph.


In [ ]:
records = [
    {'id': 'U-001', 'department': 'SOC', 'manager': 'N/A', 'notes': ''},
    {'id': 'U-002', 'department': 'GRC', 'manager': 'null', 'notes': 'none'},
    {'id': 'U-003', 'department': 'na', 'manager': 'Jane Smith', 'notes': '-'}
]
config = TransformerProviderConfig(name='normalize_nulls', args={
    'fields': ['department', 'manager', 'notes'],
    'null_like': ['', 'n/a', 'na', 'null', 'none', '-']
})
transformer = NormalizeNullsProvider(config)
result = transformer.transform(records)
print("NormalizeNullsProvider:")
for r in result:
    print(r)

## Case Normalizer

Standardize field values to a consistent case. Ensures `'admin'`, `'Admin'`, and `'ADMIN'`
all resolve to the same node label or property value.


In [ ]:
records = [
    {'control': 'Access Control', 'framework': 'nist csf', 'status': 'Active'},
    {'control': 'Data Protection', 'framework': 'SOC 2', 'status': 'pending'}
]
config = TransformerProviderConfig(name='to_upper', args={'mode': 'upper'})
transformer = NormalizeCaseProvider(config)
result = transformer.transform(records)
print("NormalizeCaseProvider (mode=upper):")
for r in result:
    print(r)

print()
config = TransformerProviderConfig(name='to_lower', args={'mode': 'lower'})
transformer = NormalizeCaseProvider(config)
result = transformer.transform(records)
print("NormalizeCaseProvider (mode=lower):")
for r in result:
    print(r)

## Enum Normalizer

Map variant values (abbreviations, legacy codes, typos) to canonical enum labels.
For example: `'NY'`, `'new york'`, `'New York'` → `'NEW_YORK'`.


In [ ]:
records = [
    {'id': 1, 'risk_level': 'Critical'},
    {'id': 2, 'risk_level': 'HIGH'},
    {'id': 3, 'risk_level': 'Med'},
    {'id': 4, 'risk_level': 'low'},
    {'id': 5, 'risk_level': 'informational'}
]
config = TransformerProviderConfig(name='normalize_risk', args={
    'field': 'risk_level',
    'enum_map': {
        'critical': 'CRITICAL',
        'high': 'HIGH',
        'med': 'MEDIUM',
        'medium': 'MEDIUM',
        'low': 'LOW',
        'informational': 'INFO'
    },
    'case_insensitive': True
})
transformer = NormalizeEnumProvider(config)
result = transformer.transform(records)
print("NormalizeEnumProvider:")
for r in result:
    print(f"  {r['id']}: {r['risk_level']}")

## Spelling Normalizer

Correct common misspellings in field values using a configurable dictionary.
Prevents duplicate nodes caused by typos (e.g., `'Enginering'` vs `'Engineering'`).


In [ ]:
records = [
    {'id': 'F-001', 'description': 'Unathorized acess to prodution server detectd'},
    {'id': 'F-002', 'description': 'Privilage escalaton attemp from externel IP'}
]
config = TransformerProviderConfig(name='fix_spelling', args={'fields': ['description']})
transformer = NormalizeSpellingProvider(config)
try:
    result = transformer.transform(records)
    print("NormalizeSpellingProvider:")
    for r in result:
        print(f"  {r['id']}: {r['description']}")
except ImportError as e:
    print(f"NormalizeSpellingProvider: Skipped (TextBlob not installed) - {e}")

## Timestamp Normalizer

Parse varied date/time formats (`'06/30/2026'`, `'2026-06-30'`, `'June 30, 2026'`) into
consistent ISO 8601 strings for reliable temporal queries.


In [ ]:
records = [
    {'event': 'auth_failure', 'created_at': 'Jun 20, 2026 14:30:45', 'resolved_at': '06/21/2026 09:00'},
    {'event': 'data_breach', 'created_at': '2026-06-22T18:45:00Z', 'resolved_at': '22 June 2026 20:30'},
    {'event': 'policy_violation', 'created_at': '06-23-2026 11:15:30', 'resolved_at': '2026.06.23 15:00:00'}
]
config = TransformerProviderConfig(name='normalize_timestamps', args={
    'fields': ['created_at', 'resolved_at']
})
transformer = NormalizeTimestampProvider(config)
result = transformer.transform(records)
print("NormalizeTimestampProvider:")
for r in result:
    print(f"  {r['event']}: created={r['created_at']}, resolved={r['resolved_at']}")

## Summary
All normalizer providers tested successfully:
- **NormalizeWhitespaceProvider**: Strip/collapse whitespace in specified fields
- **NormalizeNullsProvider**: Convert null-like strings to None
- **NormalizeCaseProvider**: Apply case transformation (lower/upper) to all string fields
- **NormalizeEnumProvider**: Map values to standardized enums via enum_map
- **NormalizeSpellingProvider**: Correct spelling via TextBlob
- **NormalizeTimestampProvider**: Parse various date formats to ISO 8601